# Fine-tune Laya on the synthetic Kannada set (#14)

Runs `training/finetune_laya.py`: Laya's multilingual checkpoint (at Laya's reviewed commit) is fine-tuned with Laya's own training loop on the 2,874 caller sentences in `Dhwanidesk/dhwani-desk-data`, asked the eight questions the backend asks Laya, worded exactly as it asks them. It is then exported to ONNX with the upstream exporter, checked against PyTorch, and uploaded to `Dhwanidesk/dhwani-desk-laya` with a tag the backend pins.

**Before running**
- Kaggle: Settings → Accelerator **GPU T4 x2** (one GPU is used), Internet **on**; Add-ons → Secrets → `HF_TOKEN` (a **write** token for the Dhwanidesk organization).
- Colab: Runtime → **T4 GPU**; Secrets (key icon) → `HF_TOKEN`, notebook access on.

Run it with **Save Version → Save & Run All**, so it keeps running when the browser tab closes.

Step 1 is a 10-minute smoke run through every step except the upload, so an export problem shows up before the long run. Step 2 is the real run, about 1–2 hours.

In [ ]:
import os

BRANCH = 'feat/14-finetune-laya'  # 'main' once PR #46 has merged
# Start clean from the notebook's working folder, so running this cell again re-clones.
WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
os.chdir(WORK)
!rm -rf dhwani-desk
!git clone -q --depth 1 -b {BRANCH} https://github.com/Pranav-0710/dhwani-desk.git
os.chdir(f'{WORK}/dhwani-desk/training')
assert os.path.exists('finetune_laya.py'), f'finetune_laya.py is not on branch {BRANCH}'
# The versions the backend runs (backend/uv.lock), so the graph is exported and checked the
# same way. laya trains; onnx, onnxscript and onnxruntime export and check the graph.
!pip install -q 'laya==0.4.6' 'transformers==5.20.2' 'onnx==1.23.2' 'onnxscript==0.7.2' 'onnxruntime==1.31.0' huggingface_hub

In [ ]:
import os

import torch

try:
    from kaggle_secrets import UserSecretsClient

    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except ImportError:
    from google.colab import userdata

    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# Training on the CPU would take days: stop here instead.
assert torch.cuda.is_available(), 'No GPU: set the accelerator to GPU T4 x2'
# The exporter needs torch 2.6 or newer (torch.onnx.export with dynamo).
assert tuple(int(p) for p in torch.__version__.split('.')[:2]) >= (2, 6), torch.__version__
print(torch.__version__, torch.cuda.get_device_name(0))


def run(args):
    # `!python` alone carries on to the next cell when the script fails; this stops Run all.
    shell = get_ipython()
    shell.system(f'python finetune_laya.py {args}')
    if shell.user_ns['_exit_code']:
        raise RuntimeError(f'finetune_laya.py {args} failed; see the output above')

## Step 1: smoke run, 64 rows, no upload

Trains one epoch on 64 rows, exports and verifies the ONNX graph. `--tag-dataset` tags the dataset repo's current head `v1.0` if that tag does not exist yet, so this run and the real one train on the same, pinned data (#12).

In [ ]:
run('--limit 64 --epochs 1 --tag-dataset --out out/smoke')

## Step 2: full run and upload

Set `TAG` to a tag the model repo does not have yet. The script refuses to overwrite one. When it finishes, set `DECIDER=laya` and `LAYA_CHECKPOINT_REVISION` to the tag in the backend's `.env`.

Before training it checks that `HF_TOKEN` can reach the model repo and the tag is free. After exporting it loads the graph as the backend does and scores it on the seed set; it publishes only if it gets at least half the departments right and does not give every sentence the same one.

In [ ]:
TAG = 'kn-ft-v1'
run(f'--out out/full --push-tag {TAG}')

In [ ]:
import json

run_info = json.load(open('out/full/run.json', encoding='utf-8'))
print('rows', run_info['rows'], '| minutes', run_info['minutes'], '| loss per epoch', run_info['summary']['epoch_loss'])
print(json.dumps(json.load(open('out/full/publish/load_check.json', encoding='utf-8')), indent=1))